# LeetCode 1090 — Largest Values From Labels bằng Python

Link đề: https://leetcode.com/problems/largest-values-from-labels/ · Độ khó: **Medium** · Chủ đề: Greedy (tham lam), Sorting (sắp xếp), Hash Table (bảng băm), Heap (hàng đợi ưu tiên), Counting (đếm)

Skeleton LeetCode:

```python
class Solution:
    def largestValsFromLabels(self, values: List[int], labels: List[int], numWanted: int, useLimit: int) -> int:
        ...
```

Ghi chú: có `n` món đồ, món `i` có giá trị `values[i]` và nhãn `labels[i]`. Chọn một **tập con** các món sao cho **tổng giá trị lớn nhất** với hai ràng buộc: số món được chọn **không quá** `numWanted`, và số món **cùng một nhãn** được chọn **không quá** `useLimit`. Trả về tổng lớn nhất. Vì mọi giá trị đều **không âm**, chọn thêm món không bao giờ làm tổng giảm; ý tưởng chính: **tham lam**: sắp xếp các món theo giá trị **giảm dần**, duyệt lần lượt và chọn món nào nếu nhãn của nó **chưa dùng quá `useLimit`** lần, dừng khi đã chọn đủ `numWanted` món.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề và vì sao với **giá trị không âm** thì nên chọn **nhiều** món (đến `numWanted`) và ưu tiên món giá trị lớn.
2. Nắm các công cụ Python cần thiết: `zip`, `sorted(..., reverse=True)` trên cặp, `dict` đếm (`get`, `defaultdict`), `heapq` (`heappop`, `nlargest`), `itertools.combinations`.
3. Hiểu **tham lam trên hai ràng buộc**: một ràng buộc tổng (`numWanted`) và một ràng buộc theo nhóm (`useLimit` cho mỗi nhãn); chứng minh bằng lập luận trao đổi.
4. Đi qua 4 cách: thử mọi tập con hợp lệ (brute force, dùng làm oracle), sắp xếp rồi duyệt tham lam (cách nộp chính), heap lấy lần lượt phần tử lớn nhất và gom theo nhãn lấy `useLimit` món lớn nhất mỗi nhãn rồi lấy `numWanted` món lớn nhất.
5. Tránh các lỗi kinh điển: bỏ qua `useLimit`, bỏ qua `numWanted`, sắp xếp `values` và `labels` **riêng rẽ** (làm lệch cặp), lệch một khi so sánh `used < useLimit`, và sắp xếp theo nhãn thay vì theo giá trị.

## 2. Đề bài & ví dụ

Cho `n` món đồ với giá trị và nhãn là hai mảng số nguyên `values` và `labels`; cho thêm hai số nguyên `numWanted` và `useLimit`.

Tìm một **tập con** các món có **tổng giá trị lớn nhất** thỏa:

- Số món **không quá** `numWanted`.
- Số món có **cùng một nhãn** **không quá** `useLimit`.

Trả về tổng lớn nhất.

**Ví dụ 1:** `values = [5,4,3,2,1]`, `labels = [1,1,2,2,3]`, `numWanted = 3`, `useLimit = 1` -> `9`. Chọn món thứ nhất, thứ ba và thứ năm: `5 + 3 + 1`.

**Ví dụ 2:** `values = [5,4,3,2,1]`, `labels = [1,3,3,3,2]`, `numWanted = 3`, `useLimit = 2` -> `12`. Chọn món thứ nhất, thứ hai và thứ ba: `5 + 4 + 3`.

**Ví dụ 3:** `values = [9,8,8,7,6]`, `labels = [0,0,0,1,1]`, `numWanted = 3`, `useLimit = 1` -> `16`. Chọn món thứ nhất và thứ tư: `9 + 7`.

**Ràng buộc:**

- `n == values.length == labels.length`
- `1 <= n <= 2 * 10^4`
- `0 <= values[i], labels[i] <= 2 * 10^4`
- `1 <= numWanted, useLimit <= n`

In [1]:
from typing import List
from collections import defaultdict
from itertools import combinations
import heapq, random, time

examples = [
    ([5, 4, 3, 2, 1], [1, 1, 2, 2, 3], 3, 1, 9),
    ([5, 4, 3, 2, 1], [1, 3, 3, 3, 2], 3, 2, 12),
    ([9, 8, 8, 7, 6], [0, 0, 0, 1, 1], 3, 1, 16),
]
for values, labels, nw, ul, expected in examples:
    print(f"values={values} labels={labels} numWanted={nw} useLimit={ul} -> kỳ vọng {expected}")

values=[5, 4, 3, 2, 1] labels=[1, 1, 2, 2, 3] numWanted=3 useLimit=1 -> kỳ vọng 9
values=[5, 4, 3, 2, 1] labels=[1, 3, 3, 3, 2] numWanted=3 useLimit=2 -> kỳ vọng 12
values=[9, 8, 8, 7, 6] labels=[0, 0, 0, 1, 1] numWanted=3 useLimit=1 -> kỳ vọng 16


## 3. Kiến thức nền

### 3.1. Ghép cặp và sắp xếp

`zip(values, labels)` ghép từng phần tử thành cặp `(value, label)`. `sorted(pairs, reverse=True)` sắp xếp cặp theo giá trị **giảm dần** (khi hòa giá trị thì theo nhãn giảm dần, không ảnh hưởng đến đáp án). Sắp xếp `values` và `labels` **riêng rẽ** sẽ làm mất cặp tương ứng.

In [2]:
values, labels = [5, 4, 3, 2, 1], [1, 3, 3, 3, 2]
pairs = sorted(zip(values, labels), reverse=True)
print(pairs)
print("sai: sort riêng rẽ ->", list(zip(sorted(values, reverse=True), sorted(labels, reverse=True))))

[(5, 1), (4, 3), (3, 3), (2, 3), (1, 2)]
sai: sort riêng rẽ -> [(5, 3), (4, 3), (3, 3), (2, 2), (1, 1)]


### 3.2. Đếm theo nhãn

`used = {}` và `used.get(label, 0)` lấy số lần đã dùng nhãn (mặc định `0`); `defaultdict(int)` tự khởi tạo `0`. Điều kiện còn được dùng nhãn: `used[label] < useLimit`.

In [3]:
used = defaultdict(int)
for lab in [1, 1, 2, 1]:
    print(lab, "đã dùng", used[lab], "lần", "-> còn được dùng" if used[lab] < 2 else "-> hết lượt")
    used[lab] += 1

1 đã dùng 0 lần -> còn được dùng
1 đã dùng 1 lần -> còn được dùng
2 đã dùng 0 lần -> còn được dùng
1 đã dùng 2 lần -> hết lượt


### 3.3. `heapq`

`heapq` là min-heap; để lấy phần tử lớn nhất ta lưu số âm. `heapq.nlargest(k, iterable)` trả về `k` phần tử lớn nhất theo thứ tự giảm dần.

In [4]:
h = [-v for v in [5, 1, 4]]
heapq.heapify(h)
print(-heapq.heappop(h), -heapq.heappop(h))
print(heapq.nlargest(2, [5, 1, 4, 9, 3]))

5 4
[9, 5]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 3: `values = [9,8,8,7,6]`, `labels = [0,0,0,1,1]`, `numWanted = 3`, `useLimit = 1`.

- Sắp xếp theo giá trị giảm dần: `(9, nhãn 0)`, `(8, 0)`, `(8, 0)`, `(7, 1)`, `(6, 1)`.
- Duyệt: `9` (nhãn `0` chưa dùng) ⇒ chọn, `đã chọn = 1`, tổng `9`. `8` (nhãn `0` đã dùng `1 = useLimit`) ⇒ bỏ. `8` nữa ⇒ bỏ. `7` (nhãn `1` chưa dùng) ⇒ chọn, `đã chọn = 2`, tổng `16`. `6` (nhãn `1` đã dùng đủ) ⇒ bỏ.
- Tổng `16` ✓ (chỉ chọn được `2` món dù `numWanted = 3`).

Ví dụ 2: `values = [5,4,3,2,1]`, `labels = [1,3,3,3,2]`, `numWanted = 3`, `useLimit = 2`: duyệt `5` (nhãn `1`) chọn; `4` (nhãn `3`) chọn; `3` (nhãn `3`, đã dùng `1 < 2`) chọn: đủ `3` món, tổng `12` ✓.

## 5. Cách 1 — Brute Force: thử mọi tập con hợp lệ

**Ý tưởng.** Với `n` món có `2^n` tập con. Với mỗi tập con, kiểm tra hai ràng buộc (số món `<= numWanted` và mỗi nhãn xuất hiện `<= useLimit` lần) rồi lấy tổng lớn nhất. Đúng nguyên văn đề, không dùng tham lam nên là **oracle độc lập** (dùng được với `n <= 15`).

In [5]:
def largest_brute(values: List[int], labels: List[int], numWanted: int, useLimit: int) -> int:
    n = len(values)
    best = 0                                              # tập rỗng có tổng 0 (mọi giá trị không âm)
    for mask in range(1 << n):
        chosen = [i for i in range(n) if mask >> i & 1]
        if len(chosen) > numWanted:
            continue
        count = {}
        ok = True
        for i in chosen:
            count[labels[i]] = count.get(labels[i], 0) + 1
            if count[labels[i]] > useLimit:
                ok = False
                break
        if ok:
            best = max(best, sum(values[i] for i in chosen))
    return best

for values, labels, nw, ul, expected in examples:
    got = largest_brute(values, labels, nw, ul)
    print(f"brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute = 9 OK
brute = 12 OK
brute = 16 OK


**Độ phức tạp và nút thắt.** `O(2^n * n)`: chỉ dùng được với `n` rất nhỏ. Đo thử:

In [6]:
rng = random.Random(1090)
for n in (10, 12, 14, 16):
    values = [rng.randint(0, 20) for _ in range(n)]
    labels = [rng.randint(0, 3) for _ in range(n)]
    t0 = time.perf_counter()
    largest_brute(values, labels, n // 2, 2)
    dt = time.perf_counter() - t0
    print(f"n={n:2}: {dt * 1000:9.1f} ms")

n=10:       0.9 ms
n=12:       4.3 ms
n=14:      20.1 ms
n=16:      80.5 ms


Nút thắt: ta xét cả những tập con mà chắc chắn không tốt (bỏ món lớn, giữ món nhỏ). Vì giá trị không âm nên luôn nên ưu tiên món **lớn hơn**.

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam theo giá trị giảm dần.** Sắp xếp các món theo giá trị giảm dần. Duyệt từng món: nếu số món đã chọn `< numWanted` **và** nhãn của món còn lượt (`used[label] < useLimit`) thì chọn món đó. Dừng khi đã chọn đủ `numWanted` món.

**Vì sao đúng (tóm tắt).** Các tập con hợp lệ (tối đa `numWanted` món, mỗi nhãn tối đa `useLimit` món) có **tính chất trao đổi**: nếu `A` và `B` hợp lệ và `|A| < |B|` thì có món `x` trong `B` mà không trong `A` sao cho `A + x` vẫn hợp lệ. Với tính chất này, tham lam theo giá trị giảm dần là tối ưu khi mọi giá trị không âm; chứng minh đầy đủ ở mục 10.

| Cách | Ý tưởng |
|---|---|
| 2 | Sắp xếp cặp theo giá trị giảm dần, duyệt và đếm nhãn (nộp) |
| 3 | Heap: lấy lần lượt món lớn nhất, kiểm tra nhãn, dừng khi đủ `numWanted` |
| 4 | Gom theo nhãn, mỗi nhãn giữ `useLimit` món lớn nhất, rồi lấy `numWanted` món lớn nhất trong tất cả |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Sắp xếp rồi duyệt tham lam (cách nộp chính)

**Giải thích từng đoạn.**

- `items = sorted(zip(values, labels), reverse=True)`: các món theo giá trị giảm dần.
- `used`: đếm số món đã chọn theo nhãn; `taken`: số món đã chọn; `total`: tổng.
- Với mỗi `(v, l)`: nếu `taken == numWanted` thì dừng; nếu `used[l] < useLimit` thì chọn (cộng `v`, tăng `used[l]` và `taken`).

In [7]:
def largest_sorted(values: List[int], labels: List[int], numWanted: int, useLimit: int) -> int:
    items = sorted(zip(values, labels), reverse=True)
    used = {}
    total = taken = 0
    for v, l in items:
        if taken == numWanted:
            break
        if used.get(l, 0) < useLimit:
            used[l] = used.get(l, 0) + 1
            total += v
            taken += 1
    return total

for values, labels, nw, ul, expected in examples:
    got = largest_sorted(values, labels, nw, ul)
    print(f"sắp xếp = {got}", "OK" if got == expected else "SAI")
    assert got == expected

sắp xếp = 9 OK
sắp xếp = 12 OK
sắp xếp = 16 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def largestValsFromLabels(self, values: List[int], labels: List[int], numWanted: int, useLimit: int) -> int:
        items = sorted(zip(values, labels), reverse=True)    # giá trị giảm dần, giữ nguyên cặp (giá trị, nhãn)
        used = {}                                            # used[l]: số món nhãn l đã chọn
        total = 0                                            # tổng giá trị đã chọn
        taken = 0                                            # số món đã chọn
        for v, l in items:
            if taken == numWanted:                           # đã đủ số món
                break
            if used.get(l, 0) < useLimit:                    # nhãn này còn lượt
                used[l] = used.get(l, 0) + 1
                total += v
                taken += 1
        return total

sol = lambda v, l, nw, ul: Solution().largestValsFromLabels(v, l, nw, ul)
for values, labels, nw, ul, expected in examples:
    assert sol(values, labels, nw, ul) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** sắp xếp `O(n log n)`, duyệt `O(n)`; bộ nhớ `O(n)`.

---

### 7.2. Cách 3 — Heap

**Ý tưởng khác.** Thay vì sắp xếp toàn bộ, đưa mọi món vào max-heap (số âm) rồi lấy lần lượt món lớn nhất; kiểm tra nhãn như Cách 2. Nếu `numWanted` nhỏ thì ta dừng sớm sau khi lấy đủ; việc dựng heap tốn `O(n)`, mỗi lần `heappop` tốn `O(log n)`.

In [9]:
def largest_heap(values: List[int], labels: List[int], numWanted: int, useLimit: int) -> int:
    heap = [(-v, l) for v, l in zip(values, labels)]
    heapq.heapify(heap)
    used = {}
    total = taken = 0
    while heap and taken < numWanted:
        neg_v, l = heapq.heappop(heap)
        if used.get(l, 0) < useLimit:
            used[l] = used.get(l, 0) + 1
            total -= neg_v
            taken += 1
    return total

for values, labels, nw, ul, expected in examples:
    got = largest_heap(values, labels, nw, ul)
    print(f"heap = {got}", "OK" if got == expected else "SAI")
    assert got == expected

heap = 9 OK
heap = 12 OK
heap = 16 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def largestValsFromLabels(self, values: List[int], labels: List[int], numWanted: int, useLimit: int) -> int:
        import heapq
        heap = [(-v, l) for v, l in zip(values, labels)]      # max-heap theo giá trị (lưu số âm)
        heapq.heapify(heap)
        used = {}
        total = taken = 0
        while heap and taken < numWanted:                     # dừng khi đủ số món hoặc hết món
            neg_v, l = heapq.heappop(heap)                    # món có giá trị lớn nhất còn lại
            if used.get(l, 0) < useLimit:
                used[l] = used.get(l, 0) + 1
                total -= neg_v
                taken += 1
        return total

sol_heap = lambda v, l, nw, ul: Solution().largestValsFromLabels(v, l, nw, ul)
for values, labels, nw, ul, expected in examples:
    assert sol_heap(values, labels, nw, ul) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** dựng heap `O(n)`, mỗi lần lấy `O(log n)`, tối đa `n` lần: `O(n log n)` trong trường hợp xấu (khi phải bỏ qua nhiều món vì hết lượt nhãn); bộ nhớ `O(n)`.

---

### 7.3. Cách 4 — Gom theo nhãn rồi lấy `numWanted` món lớn nhất

**Ý tưởng khác.** Mỗi nhãn chỉ có thể đóng góp **tối đa `useLimit` món**, và nếu đóng góp thì nên là các món **lớn nhất** của nhãn đó. Nên: gom giá trị theo nhãn, sắp xếp từng nhóm giảm dần và giữ `useLimit` giá trị đầu; gộp tất cả các giá trị đã giữ vào một danh sách `pool`; đáp án là tổng của `numWanted` phần tử lớn nhất trong `pool` (nếu `pool` ít hơn thì lấy hết).

In [11]:
def largest_group(values: List[int], labels: List[int], numWanted: int, useLimit: int) -> int:
    groups = defaultdict(list)
    for v, l in zip(values, labels):
        groups[l].append(v)
    pool = []
    for vs in groups.values():
        vs.sort(reverse=True)
        pool.extend(vs[:useLimit])                          # mỗi nhãn chỉ giữ useLimit món lớn nhất
    return sum(heapq.nlargest(numWanted, pool))

for values, labels, nw, ul, expected in examples:
    got = largest_group(values, labels, nw, ul)
    print(f"gom theo nhãn = {got}", "OK" if got == expected else "SAI")
    assert got == expected

gom theo nhãn = 9 OK
gom theo nhãn = 12 OK
gom theo nhãn = 16 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def largestValsFromLabels(self, values: List[int], labels: List[int], numWanted: int, useLimit: int) -> int:
        import heapq
        from collections import defaultdict
        groups = defaultdict(list)                           # groups[l]: các giá trị của nhãn l
        for v, l in zip(values, labels):
            groups[l].append(v)
        pool = []                                            # các món "ứng viên": mỗi nhãn giữ useLimit món lớn nhất
        for vs in groups.values():
            vs.sort(reverse=True)
            pool.extend(vs[:useLimit])
        return sum(heapq.nlargest(numWanted, pool))          # lấy numWanted món lớn nhất trong các ứng viên

sol_group = lambda v, l, nw, ul: Solution().largestValsFromLabels(v, l, nw, ul)
for values, labels, nw, ul, expected in examples:
    assert sol_group(values, labels, nw, ul) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n log n)` thời gian (sắp xếp các nhóm), bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 3

In từng món được xét, trạng thái `used`, `taken`, `total`.

In [13]:
def trace_sorted(values, labels, numWanted, useLimit):
    items = sorted(zip(values, labels), reverse=True)
    print("các món (giá trị, nhãn) theo giá trị giảm dần:", items)
    used, total, taken = {}, 0, 0
    for v, l in items:
        if taken == numWanted:
            print(f"  đã đủ {numWanted} món -> dừng")
            break
        if used.get(l, 0) < useLimit:
            used[l] = used.get(l, 0) + 1
            total += v
            taken += 1
            print(f"  ({v}, nhãn {l}): chọn; used = {used}, taken = {taken}, total = {total}")
        else:
            print(f"  ({v}, nhãn {l}): nhãn đã dùng {used[l]} = useLimit -> bỏ")
    print("tổng =", total)
    return total

assert trace_sorted([9, 8, 8, 7, 6], [0, 0, 0, 1, 1], 3, 1) == 16

các món (giá trị, nhãn) theo giá trị giảm dần: [(9, 0), (8, 0), (8, 0), (7, 1), (6, 1)]
  (9, nhãn 0): chọn; used = {0: 1}, taken = 1, total = 9
  (8, nhãn 0): nhãn đã dùng 1 = useLimit -> bỏ
  (8, nhãn 0): nhãn đã dùng 1 = useLimit -> bỏ
  (7, nhãn 1): chọn; used = {0: 1, 1: 1}, taken = 2, total = 16
  (6, nhãn 1): nhãn đã dùng 1 = useLimit -> bỏ
tổng = 16


### 8.2. Cách 4 cho ví dụ 2

In nhóm theo nhãn, `pool` và kết quả `nlargest`.

In [14]:
values, labels, nw, ul = [5, 4, 3, 2, 1], [1, 3, 3, 3, 2], 3, 2
groups = defaultdict(list)
for v, l in zip(values, labels):
    groups[l].append(v)
print("gom theo nhãn:", dict(groups))
pool = []
for l, vs in groups.items():
    vs.sort(reverse=True)
    print(f"  nhãn {l}: {vs} -> giữ {vs[:ul]}")
    pool.extend(vs[:ul])
print("pool =", pool, "-> nlargest(3) =", heapq.nlargest(nw, pool), "tổng =", sum(heapq.nlargest(nw, pool)))
assert sum(heapq.nlargest(nw, pool)) == 12

gom theo nhãn: {1: [5], 3: [4, 3, 2], 2: [1]}
  nhãn 1: [5] -> giữ [5]
  nhãn 3: [4, 3, 2] -> giữ [4, 3]
  nhãn 2: [1] -> giữ [1]
pool = [5, 4, 3, 1] -> nlargest(3) = [5, 4, 3] tổng = 12


### 8.3. Vì sao không được sắp xếp `values` và `labels` riêng rẽ

Sắp xếp riêng làm giá trị lớn nhất bị ghép với nhãn khác.

In [15]:
values, labels, nw, ul = [9, 8, 8, 7, 6], [0, 0, 0, 1, 1], 3, 1
wrong_pairs = list(zip(sorted(values, reverse=True), sorted(labels, reverse=True)))
print("cặp sau khi sắp xếp riêng:", wrong_pairs, "| cặp đúng:", sorted(zip(values, labels), reverse=True))

cặp sau khi sắp xếp riêng: [(9, 1), (8, 1), (8, 0), (7, 0), (6, 0)] | cặp đúng: [(9, 0), (8, 0), (8, 0), (7, 1), (6, 1)]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (sắp xếp rồi duyệt tham lam): ngắn nhất, `O(n log n)`.

```python
class Solution:
    def largestValsFromLabels(self, values: List[int], labels: List[int], numWanted: int, useLimit: int) -> int:
        items = sorted(zip(values, labels), reverse=True)    # giá trị giảm dần, giữ nguyên cặp (giá trị, nhãn)
        used = {}                                            # used[l]: số món nhãn l đã chọn
        total = 0                                            # tổng giá trị đã chọn
        taken = 0                                            # số món đã chọn
        for v, l in items:
            if taken == numWanted:                           # đã đủ số món
                break
            if used.get(l, 0) < useLimit:                    # nhãn này còn lượt
                used[l] = used.get(l, 0) + 1
                total += v
                taken += 1
        return total
```

In [16]:
class Solution:
    def largestValsFromLabels(self, values: List[int], labels: List[int], numWanted: int, useLimit: int) -> int:
        items = sorted(zip(values, labels), reverse=True)    # giá trị giảm dần, giữ nguyên cặp (giá trị, nhãn)
        used = {}                                            # used[l]: số món nhãn l đã chọn
        total = 0                                            # tổng giá trị đã chọn
        taken = 0                                            # số món đã chọn
        for v, l in items:
            if taken == numWanted:                           # đã đủ số món
                break
            if used.get(l, 0) < useLimit:                    # nhãn này còn lượt
                used[l] = used.get(l, 0) + 1
                total += v
                taken += 1
        return total

sol = lambda v, l, nw, ul: Solution().largestValsFromLabels(v, l, nw, ul)
for values, labels, nw, ul, expected in examples:
    got = sol(values, labels, nw, ul)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

9 OK
12 OK
16 OK


## 10. Vì sao đúng

**Họ các tập hợp lệ.** Gọi `F` là họ các tập con hợp lệ: `|S| <= numWanted` và mỗi nhãn có không quá `useLimit` món trong `S`. Hai tính chất:

1. **Đóng với tập con:** bỏ bớt món khỏi tập hợp lệ vẫn hợp lệ.
2. **Tính chất trao đổi:** nếu `A, B` thuộc `F` và `|A| < |B|` thì tồn tại `x` thuộc `B` nhưng không thuộc `A` sao cho `A + x` thuộc `F`. Thật vậy, vì `|A| < |B|` nên có một nhãn `l` mà số món nhãn `l` trong `B` lớn hơn trong `A`; chọn `x` là một món nhãn `l` của `B` không thuộc `A`. Khi đó `A + x` có số món nhãn `l` bằng `|A_l| + 1 <= |B_l| <= useLimit` và tổng số món `|A| + 1 <= |B| <= numWanted`: hợp lệ.

**Tham lam tối ưu.** Sắp xếp các món theo giá trị giảm dần (hòa thì theo thứ tự cố định nào đó). Gọi `g_1 >= g_2 >= ...` là các món tham lam chọn (theo thứ tự chọn) và `S` là một tập hợp lệ bất kỳ có các món `s_1 >= s_2 >= ... >= s_m` (sắp xếp theo thứ tự đã cho). Ta chứng minh số món của tham lam `>= m` và `value(g_k) >= value(s_k)` với mọi `k <= m`; từ đó tổng của tham lam `>=` tổng của `S` (mọi giá trị không âm nên các món thừa của tham lam chỉ cộng thêm).

Giả sử vi phạm tại `k` nhỏ nhất: tham lam có ít hơn `k` món, hoặc có `g_k` với `value(g_k) < value(s_k)`. Đặt `A = {g_1, ..., g_{k-1}}` (toàn bộ món của tham lam nếu ít hơn `k`) và `B = {s_1, ..., s_k}`. Cả hai thuộc `F` (`B` là tập con của `S`) và `|A| < |B|`. Theo tính chất trao đổi có `x` thuộc `B` mà không thuộc `A` với `A + x` hợp lệ; vì `x` thuộc `B` nên `value(x) >= value(s_k) > value(g_k)` (hoặc không có `g_k`). Khi thuật toán xét đến `x` (xếp trước `g_k`), tập đã chọn là một tập con của `A` nên `chosen + x` là tập con của `A + x`, hợp lệ, do đó thuật toán chọn `x`, tức `x` thuộc `A`. Mâu thuẫn với `x` không thuộc `A`.

**Cách 4.** Với mỗi nhãn, nếu một tập tối ưu dùng `j <= useLimit` món nhãn đó thì có thể thay chúng bằng `j` món lớn nhất của nhãn (không giảm tổng, vẫn hợp lệ). Vậy mọi lựa chọn đều nằm trong `pool` (các món lớn nhất mỗi nhãn, `<= useLimit` món mỗi nhãn); trong `pool` không còn ràng buộc nhãn nào vi phạm khi chọn bất kỳ tập con nào (mỗi nhãn tối đa `useLimit` món), nên chỉ cần lấy `numWanted` món lớn nhất. **Cách 3** là Cách 2 với heap thay cho sắp xếp.

**Kiểm chứng bằng máy.** Oracle (Cách 1) thử mọi tập con. Thêm một oracle độc lập thứ hai bằng **quy hoạch động theo nhãn** (cặp `(số nhãn đã xét, số món đã chọn)`), dùng cho `n` lớn hơn. Ta đối chiếu mọi cách trên rất nhiều ví dụ ngẫu nhiên (giá trị nhỏ để có nhiều hòa).

In [17]:
def largest_dp(values: List[int], labels: List[int], numWanted: int, useLimit: int) -> int:
    """Quy hoạch động theo nhãn: dp[c] = tổng lớn nhất khi đã chọn đúng c món từ các nhãn đã xét."""
    groups = defaultdict(list)
    for v, l in zip(values, labels):
        groups[l].append(v)
    NEG = float("-inf")
    dp = [0] + [NEG] * numWanted
    for vs in groups.values():
        vs.sort(reverse=True)
        prefix = [0]
        for v in vs[:useLimit]:
            prefix.append(prefix[-1] + v)                    # prefix[j]: tổng j món lớn nhất của nhãn này
        new = dp[:]
        for c in range(numWanted + 1):
            if dp[c] == NEG:
                continue
            for j in range(1, len(prefix)):
                if c + j <= numWanted:
                    new[c + j] = max(new[c + j], dp[c] + prefix[j])
        dp = new
    return max(x for x in dp if x != NEG)

impls = {"sắp xếp": largest_sorted, "heap": largest_heap, "gom nhãn": largest_group, "nộp (Cách 2)": sol,
         "Cách 3 (nộp)": sol_heap, "Cách 4 (nộp)": sol_group, "DP nhãn": largest_dp}
rng = random.Random(10)
total = 0
for _ in range(4000):
    n = rng.randint(1, 12)
    values = [rng.randint(0, rng.choice([2, 5, 20])) for _ in range(n)]
    labels = [rng.randint(0, rng.choice([1, 2, 4])) for _ in range(n)]
    nw, ul = rng.randint(1, n), rng.randint(1, n)
    ref = largest_brute(values, labels, nw, ul)
    for name, fn in impls.items():
        assert fn(values[:], labels[:], nw, ul) == ref, (name, values, labels, nw, ul)
    total += 1
print("đã đối chiếu", total, "ví dụ ngẫu nhiên: mọi cách khớp oracle (tập con) và DP")

đã đối chiếu 4000 ví dụ ngẫu nhiên: mọi cách khớp oracle (tập con) và DP


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi tập con | `O(2^n * n)` | `O(n)` |
| 2. Sắp xếp + tham lam | `O(n log n)` | `O(n)` |
| 3. Heap | `O(n + k log n)` (`k` số lần lấy) | `O(n)` |
| 4. Gom theo nhãn + `nlargest` | `O(n log n)` | `O(n)` |

Đo ở cỡ tối đa (`n = 2 * 10^4`) với các dạng: ngẫu nhiên, một nhãn duy nhất, mọi nhãn khác nhau, `numWanted` nhỏ, `useLimit` nhỏ:

In [18]:
rng = random.Random(11)
n = 2 * 10 ** 4
cases = {
    "ngẫu nhiên, 100 nhãn": ([rng.randint(0, 2 * 10 ** 4) for _ in range(n)], [rng.randint(0, 99) for _ in range(n)], 5000, 30),
    "một nhãn duy nhất": ([rng.randint(0, 2 * 10 ** 4) for _ in range(n)], [7] * n, 10000, 3000),
    "mọi nhãn khác nhau": ([rng.randint(0, 2 * 10 ** 4) for _ in range(n)], list(range(n)), 8000, 1),
    "numWanted = 1": ([rng.randint(0, 2 * 10 ** 4) for _ in range(n)], [rng.randint(0, 5) for _ in range(n)], 1, n),
    "useLimit = 1, ít nhãn": ([rng.randint(0, 2 * 10 ** 4) for _ in range(n)], [rng.randint(0, 3) for _ in range(n)], n, 1),
    "mọi giá trị bằng nhau": ([5] * n, [rng.randint(0, 50) for _ in range(n)], 777, 20),
}
for label, (values, labels, nw, ul) in cases.items():
    ref = sol(values, labels, nw, ul)
    row = [f"{label:26}"]
    for name, fn in (("sắp xếp", sol), ("heap", sol_heap), ("gom nhãn", sol_group)):
        t0 = time.perf_counter()
        got = fn(values, labels, nw, ul)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.2f} ms")
    print(" | ".join(row), "| đáp án", ref)

ngẫu nhiên, 100 nhãn       | sắp xếp   26.44 ms | heap   10.88 ms | gom nhãn    8.60 ms | đáp án 55301952
một nhãn duy nhất          | sắp xếp    9.02 ms | heap    8.54 ms | gom nhãn    8.47 ms | đáp án 55470075


mọi nhãn khác nhau         | sắp xếp    9.66 ms | heap   11.26 ms | gom nhãn    8.65 ms | đáp án 127929947
numWanted = 1              | sắp xếp    6.94 ms | heap    6.83 ms | gom nhãn    6.65 ms | đáp án 20000
useLimit = 1, ít nhãn      | sắp xếp    7.96 ms | heap    7.77 ms | gom nhãn    7.70 ms | đáp án 79989
mọi giá trị bằng nhau      | sắp xếp    5.76 ms | heap    5.72 ms | gom nhãn    5.72 ms | đáp án 3885


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), test ngẫu nhiên có seed (đúng ràng buộc), test cỡ lớn và các tính chất.

In [19]:
def check_all(values, labels, nw, ul, expected=None, brute=True):
    results = {name: fn(values[:], labels[:], nw, ul) for name, fn in impls.items()}
    first = results["nộp (Cách 2)"]
    for name, got in results.items():
        assert got == first, (name, got, first, values[:10], labels[:10], nw, ul)
    if brute and len(values) <= 13:
        assert largest_brute(values, labels, nw, ul) == first, (values, labels, nw, ul)
    if expected is not None:
        assert first == expected, (values, labels, nw, ul, first, expected)
    return first

# 12.1 ví dụ đề
for v, l, nw, ul, e in examples:
    check_all(v, l, nw, ul, e)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([5], [0], 1, 1, 5),                                # một món
    ([0, 0, 0], [1, 2, 3], 3, 1, 0),                    # mọi giá trị 0
    ([3, 2, 1], [1, 1, 1], 3, 2, 5),                    # một nhãn, useLimit = 2: chỉ lấy hai món lớn nhất
    ([1, 2, 3], [1, 2, 3], 2, 1, 5),                    # numWanted = 2: hai món lớn nhất khác nhãn
    ([10, 1], [7, 7], 2, 1, 10),                        # cùng nhãn, useLimit = 1: chỉ món lớn nhất
    ([4, 4, 4, 4], [1, 2, 1, 2], 3, 1, 8),              # useLimit = 1 với hai nhãn: chỉ chọn được hai món
    ([4, 4, 4, 4], [1, 2, 1, 2], 4, 2, 16),            # đủ cả bốn món
    ([9, 8, 7, 6], [1, 1, 1, 1], 1, 4, 9),              # numWanted = 1: món lớn nhất
    ([9, 8, 7, 6], [1, 1, 1, 1], 4, 4, 30),             # lấy tất cả
    ([5, 4, 3, 2, 1], [1, 1, 2, 2, 3], 5, 5, 15),       # không ràng buộc: tổng tất cả
    ([5, 4, 3, 2, 1], [1, 1, 2, 2, 3], 1, 1, 5),        # chỉ một món
    ([10, 9, 8, 7, 6, 5], [1, 1, 2, 2, 3, 3], 4, 1, 24),    # tính tay ở bài tập 2: chỉ lấy được 3 món (mỗi nhãn một)
    ([20000] * 4, [0, 1, 2, 3], 4, 1, 80000),           # giá trị lớn nhất
    ([7, 7, 7], [0, 0, 0], 2, 2, 14),                   # giá trị bằng nhau
]
for v, l, nw, ul, e in edge:
    check_all(v, l, nw, ul, e)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 oracle DP và oracle tập con đồng ý trên ví dụ có nhiều hòa
rng = random.Random(123)
for _ in range(300):
    n = rng.randint(1, 12)
    v = [rng.randint(0, 3) for _ in range(n)]
    l = [rng.randint(0, 2) for _ in range(n)]
    nw, ul = rng.randint(1, n), rng.randint(1, n)
    assert largest_dp(v[:], l[:], nw, ul) == largest_brute(v, l, nw, ul)
print("12.3 hai oracle khớp nhau: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 14 ca


12.3 hai oracle khớp nhau: OK


In [20]:
# 12.4 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_no_use_limit(values, labels, numWanted, useLimit):      # bỏ qua useLimit: lấy numWanted món lớn nhất
    return sum(sorted(values, reverse=True)[:numWanted])

def wrong_no_num_wanted(values, labels, numWanted, useLimit):     # bỏ qua numWanted: chỉ giới hạn theo nhãn
    items = sorted(zip(values, labels), reverse=True)
    used, total = {}, 0
    for v, l in items:
        if used.get(l, 0) < useLimit:
            used[l] = used.get(l, 0) + 1
            total += v
    return total

def wrong_separate_sort(values, labels, numWanted, useLimit):     # sắp xếp values và labels riêng rẽ
    vs, ls = sorted(values, reverse=True), sorted(labels, reverse=True)
    used, total, taken = {}, 0, 0
    for v, l in zip(vs, ls):
        if taken == numWanted:
            break
        if used.get(l, 0) < useLimit:
            used[l] = used.get(l, 0) + 1
            total += v
            taken += 1
    return total

def wrong_off_by_one(values, labels, numWanted, useLimit):        # dùng <= thay vì <: cho phép useLimit + 1 món mỗi nhãn
    items = sorted(zip(values, labels), reverse=True)
    used, total, taken = {}, 0, 0
    for v, l in items:
        if taken == numWanted:
            break
        if used.get(l, 0) <= useLimit:
            used[l] = used.get(l, 0) + 1
            total += v
            taken += 1
    return total

def wrong_sort_by_label(values, labels, numWanted, useLimit):     # sắp xếp theo nhãn thay vì theo giá trị
    items = sorted(zip(labels, values), reverse=True)
    used, total, taken = {}, 0, 0
    for l, v in items:
        if taken == numWanted:
            break
        if used.get(l, 0) < useLimit:
            used[l] = used.get(l, 0) + 1
            total += v
            taken += 1
    return total

def find_counter(wrong, tries=40000, seed=9):
    rng = random.Random(seed)
    best = None
    for _ in range(tries):
        n = rng.randint(1, 6)
        values = [rng.randint(0, 9) for _ in range(n)]
        labels = [rng.randint(0, 2) for _ in range(n)]
        nw, ul = rng.randint(1, n), rng.randint(1, n)
        if best is not None and n >= len(best[0]):
            continue
        if wrong(values, labels, nw, ul) != largest_brute(values, labels, nw, ul):
            best = (values, labels, nw, ul)
    return best

hand_cases = {                                       # phản ví dụ chọn tay (dễ nhìn), được xác nhận lại bằng oracle
    "bỏ qua useLimit": (wrong_no_use_limit, [5, 4, 3, 2, 1], [1, 1, 2, 2, 3], 3, 1),                 # ví dụ 1 của đề
    "bỏ qua numWanted": (wrong_no_num_wanted, [5, 4, 3, 2, 1], [1, 3, 3, 3, 2], 3, 2),                # ví dụ 2 của đề
    "sắp xếp values và labels riêng rẽ": (wrong_separate_sort, [9, 8, 8, 7, 6], [0, 0, 0, 1, 1], 3, 1),  # ví dụ 3 của đề
    "lệch một: <= useLimit": (wrong_off_by_one, [5, 4, 3, 2, 1], [1, 1, 2, 2, 3], 3, 1),
    "sắp xếp theo nhãn": (wrong_sort_by_label, [1, 9, 5], [3, 1, 2], 1, 1),                           # nhãn lớn nhất không phải giá trị lớn nhất
}
counters = {}
for name, (fn, v, l, nw, ul) in hand_cases.items():
    wrong_result = fn(v, l, nw, ul)
    right = largest_brute(v, l, nw, ul)
    assert wrong_result != right, name
    assert find_counter(fn) is not None, name           # tìm kiếm tự động cũng tìm ra phản ví dụ (nhỏ nhất)
    counters[name] = (v, l, nw, ul)
    print(f"{name:36} values={v} labels={l} numWanted={nw} useLimit={ul}: sai -> {wrong_result}, đúng -> {right}")
    check_all(v, l, nw, ul)
print("12.4 phản ví dụ: OK")

bỏ qua useLimit                      values=[5, 4, 3, 2, 1] labels=[1, 1, 2, 2, 3] numWanted=3 useLimit=1: sai -> 12, đúng -> 9


bỏ qua numWanted                     values=[5, 4, 3, 2, 1] labels=[1, 3, 3, 3, 2] numWanted=3 useLimit=2: sai -> 13, đúng -> 12


sắp xếp values và labels riêng rẽ    values=[9, 8, 8, 7, 6] labels=[0, 0, 0, 1, 1] numWanted=3 useLimit=1: sai -> 17, đúng -> 16


lệch một: <= useLimit                values=[5, 4, 3, 2, 1] labels=[1, 1, 2, 2, 3] numWanted=3 useLimit=1: sai -> 12, đúng -> 9


sắp xếp theo nhãn                    values=[1, 9, 5] labels=[3, 1, 2] numWanted=1 useLimit=1: sai -> 1, đúng -> 9
12.4 phản ví dụ: OK


In [21]:
# 12.5 ngẫu nhiên nhỏ: giá trị nhỏ để có nhiều hòa, nhiều dải số nhãn
rng = random.Random(4090)
for _ in range(3000):
    n = rng.randint(1, 13)
    values = [rng.randint(0, rng.choice([1, 3, 10, 20000])) for _ in range(n)]
    labels = [rng.randint(0, rng.choice([0, 1, 3, 20000])) for _ in range(n)]
    check_all(values, labels, rng.randint(1, n), rng.randint(1, n))
print("12.5 ngẫu nhiên nhỏ: OK")

# 12.6 ngẫu nhiên vừa (so DP) và lớn (so các cách nhanh với nhau)
for _ in range(100):
    n = rng.randint(14, 60)
    values = [rng.randint(0, rng.choice([5, 100])) for _ in range(n)]
    labels = [rng.randint(0, rng.choice([2, 6, 30])) for _ in range(n)]
    check_all(values, labels, rng.randint(1, n), rng.randint(1, n), brute=False)
for _ in range(15):
    n = rng.randint(5000, 20000)
    values = [rng.randint(0, 20000) for _ in range(n)]
    labels = [rng.randint(0, rng.choice([5, 100, 20000])) for _ in range(n)]
    nw, ul = rng.randint(1, n), rng.randint(1, n)
    results = {sol(values, labels, nw, ul), sol_heap(values, labels, nw, ul), sol_group(values, labels, nw, ul)}
    assert len(results) == 1
print("12.6 ngẫu nhiên vừa và lớn: OK")

# 12.7 tính chất: đơn điệu theo numWanted và useLimit; không phụ thuộc thứ tự món; không vượt tổng tất cả; useLimit = n cho tổng top numWanted
for _ in range(400):
    n = rng.randint(1, 15)
    values = [rng.randint(0, 10) for _ in range(n)]
    labels = [rng.randint(0, 3) for _ in range(n)]
    nw, ul = rng.randint(1, n), rng.randint(1, n)
    base = sol(values, labels, nw, ul)
    assert base <= sum(values)
    if nw < n:
        assert sol(values, labels, nw + 1, ul) >= base                                     # thêm lượt chọn không giảm tổng
    if ul < n:
        assert sol(values, labels, nw, ul + 1) >= base                                     # nới giới hạn nhãn không giảm tổng
    order = list(range(n))
    rng.shuffle(order)
    assert sol([values[i] for i in order], [labels[i] for i in order], nw, ul) == base      # thứ tự món không quan trọng
    assert sol(values, labels, nw, n) == sum(sorted(values, reverse=True)[:nw])             # useLimit = n: chỉ còn ràng buộc tổng
    assert sol(values, [0] * n, nw, ul) == sum(sorted(values, reverse=True)[:min(nw, ul)])  # một nhãn: lấy min(nw, ul) món lớn nhất
print("12.7 tính chất: OK")

12.5 ngẫu nhiên nhỏ: OK


12.6 ngẫu nhiên vừa và lớn: OK
12.7 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ qua `useLimit`

Chỉ lấy `numWanted` món lớn nhất mà quên giới hạn số món cùng nhãn.

In [22]:
v, l, nw, ul = counters["bỏ qua useLimit"]
print(f"values={v} labels={l} numWanted={nw} useLimit={ul} | sai ->", wrong_no_use_limit(v, l, nw, ul), "| đúng ->", sol(v, l, nw, ul))
assert wrong_no_use_limit(v, l, nw, ul) != sol(v, l, nw, ul)

values=[5, 4, 3, 2, 1] labels=[1, 1, 2, 2, 3] numWanted=3 useLimit=1 | sai -> 12 | đúng -> 9


Vì sao sai: ví dụ 1 chọn ba món lớn nhất `5 + 4 + 3 = 12` nhưng nhãn `1` có hai món (`5`, `4`) vượt `useLimit = 1`. **Đúng:** đếm `used[label]` và chỉ chọn khi `used[label] < useLimit`.

### Lỗi 2 — Bỏ qua `numWanted`

Chọn mọi món hợp lệ theo nhãn mà không dừng khi đủ `numWanted`.

In [23]:
v, l, nw, ul = counters["bỏ qua numWanted"]
print(f"values={v} labels={l} numWanted={nw} useLimit={ul} | sai ->", wrong_no_num_wanted(v, l, nw, ul), "| đúng ->", sol(v, l, nw, ul))
assert wrong_no_num_wanted(v, l, nw, ul) != sol(v, l, nw, ul)

values=[5, 4, 3, 2, 1] labels=[1, 3, 3, 3, 2] numWanted=3 useLimit=2 | sai -> 13 | đúng -> 12


Vì sao sai: tổng số món chọn không được vượt `numWanted`. **Đúng:** dừng khi `taken == numWanted`.

### Lỗi 3 — Sắp xếp `values` và `labels` riêng rẽ

Sau khi sắp xếp riêng, giá trị và nhãn không còn đúng cặp.

In [24]:
v, l, nw, ul = counters["sắp xếp values và labels riêng rẽ"]
print(f"values={v} labels={l} numWanted={nw} useLimit={ul} | sai ->", wrong_separate_sort(v, l, nw, ul), "| đúng ->", sol(v, l, nw, ul))
assert wrong_separate_sort(v, l, nw, ul) != sol(v, l, nw, ul)

values=[9, 8, 8, 7, 6] labels=[0, 0, 0, 1, 1] numWanted=3 useLimit=1 | sai -> 17 | đúng -> 16


Vì sao sai: mỗi món là **một cặp** `(giá trị, nhãn)`. **Đúng:** `sorted(zip(values, labels), reverse=True)`.

### Lỗi 4 — Lệch một: `<=` thay vì `<`

In [25]:
v, l, nw, ul = counters["lệch một: <= useLimit"]
print(f"values={v} labels={l} numWanted={nw} useLimit={ul} | sai ->", wrong_off_by_one(v, l, nw, ul), "| đúng ->", sol(v, l, nw, ul))
assert wrong_off_by_one(v, l, nw, ul) != sol(v, l, nw, ul)

values=[5, 4, 3, 2, 1] labels=[1, 1, 2, 2, 3] numWanted=3 useLimit=1 | sai -> 12 | đúng -> 9


Vì sao sai: nhãn đã dùng `useLimit` món thì hết lượt; điều kiện còn lượt là `used < useLimit` (không phải `<=`). **Đúng:** `if used.get(l, 0) < useLimit`.

### Lỗi 5 — Sắp xếp theo nhãn thay vì theo giá trị

In [26]:
v, l, nw, ul = counters["sắp xếp theo nhãn"]
print(f"values={v} labels={l} numWanted={nw} useLimit={ul} | sai ->", wrong_sort_by_label(v, l, nw, ul), "| đúng ->", sol(v, l, nw, ul))
assert wrong_sort_by_label(v, l, nw, ul) != sol(v, l, nw, ul)

values=[1, 9, 5] labels=[3, 1, 2] numWanted=1 useLimit=1 | sai -> 1 | đúng -> 9


Vì sao sai: tham lam phải ưu tiên **giá trị** lớn, không phải nhãn lớn. **Đúng:** khóa sắp xếp là giá trị (phần tử đầu của cặp `(value, label)`).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chọn tối đa `k` món có tổng lớn nhất, kèm giới hạn theo nhóm" với giá trị **không âm** ⇒ **tham lam**: sắp xếp theo giá trị giảm dần, đếm theo nhóm.
- Giữ cặp `(giá trị, nhãn)` cùng nhau: `zip` rồi sắp xếp.
- Dùng `dict` đếm `used[label]`; điều kiện còn lượt là `< limit`.
- Nếu **có giá trị âm**, dừng khi gặp giá trị `<= 0` (không bắt buộc chọn đủ `numWanted`).
- Có thể thay sắp xếp bằng **heap** khi chỉ cần vài phần tử lớn nhất.
- Cùng họ: Maximum Units on a Truck (1710), Assign Cookies (455), Task Scheduler (621), Top K Frequent Elements (347).
- Test nên có: một nhãn duy nhất, mọi nhãn khác nhau, `numWanted = 1`, `useLimit = 1`, mọi giá trị bằng nhau, giá trị `0`.

## 15. Biến thể

1. **Có giá trị âm:** chỉ chọn món có giá trị dương; vẫn tham lam theo giá trị giảm dần và dừng khi giá trị `<= 0`.
2. **Chọn đúng `numWanted` món** (bắt buộc, có thể có giá trị âm): không còn tham lam đơn giản; cần quy hoạch động theo nhãn.
3. **Trả về các chỉ số** được chọn: bài tập 5.
4. Bài liên quan: 1710, 455.

Kiểm biến thể 1 (có giá trị âm, chọn tối đa `numWanted`) bằng oracle tập con:

In [27]:
def largest_with_negatives(values, labels, numWanted, useLimit):
    items = sorted(zip(values, labels), reverse=True)
    used, total, taken = {}, 0, 0
    for v, l in items:
        if taken == numWanted or v <= 0:                   # giá trị không dương: chọn thêm không có lợi
            break
        if used.get(l, 0) < useLimit:
            used[l] = used.get(l, 0) + 1
            total += v
            taken += 1
    return total

def largest_with_negatives_brute(values, labels, numWanted, useLimit):
    n = len(values)
    best = 0
    for mask in range(1 << n):
        chosen = [i for i in range(n) if mask >> i & 1]
        if len(chosen) > numWanted:
            continue
        cnt = {}
        for i in chosen:
            cnt[labels[i]] = cnt.get(labels[i], 0) + 1
        if all(c <= useLimit for c in cnt.values()):
            best = max(best, sum(values[i] for i in chosen))
    return best

rng = random.Random(15)
for _ in range(2000):
    n = rng.randint(1, 10)
    values = [rng.randint(-5, 9) for _ in range(n)]
    labels = [rng.randint(0, 2) for _ in range(n)]
    nw, ul = rng.randint(1, n), rng.randint(1, n)
    assert largest_with_negatives(values, labels, nw, ul) == largest_with_negatives_brute(values, labels, nw, ul), (values, labels, nw, ul)
print("biến thể có giá trị âm: tham lam dừng ở giá trị <= 0 khớp oracle tập con")

biến thể có giá trị âm: tham lam dừng ở giá trị <= 0 khớp oracle tập con


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi tập con | Theo định nghĩa | `O(2^n n)` | `O(n)` | Oracle, `n <= 15` |
| 2. Sắp xếp + tham lam | Giá trị giảm dần, đếm nhãn | `O(n log n)` | `O(n)` | **Cách nộp chính** |
| 3. Heap | Lấy lần lượt món lớn nhất | `O(n + k log n)` | `O(n)` | Khi chỉ cần ít món |
| 4. Gom theo nhãn | `useLimit` món lớn nhất mỗi nhãn, rồi `nlargest` | `O(n log n)` | `O(n)` | Dễ chứng minh, dễ mở rộng |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(sorted(zip([3, 9, 5], [1, 2, 1]), reverse=True))
print(heapq.nlargest(2, [4, 8, 1, 8]))
print(Solution().largestValsFromLabels([3, 9, 5], [1, 2, 1], 2, 1))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # danh sách cặp đã sắp xếp
my_guess_2 = None      # hai phần tử lớn nhất
my_guess_3 = None      # kết quả largestValsFromLabels

### Bài 2 — Tính tay (dễ-vừa)

`values = [10, 9, 8, 7, 6, 5]`, `labels = [1, 1, 2, 2, 3, 3]`, `numWanted = 4`, `useLimit = 1`. Liệt kê các món được chọn theo thứ tự tham lam và tổng.

In [29]:
# Bài 2: điền các giá trị được chọn (theo thứ tự) và tổng
hand_chosen, hand_answer = None, None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành tham lam bằng sắp xếp.

In [30]:
def exercise_3(values, labels, numWanted, useLimit):
    items = sorted(zip(values, labels), reverse=True)
    used, total, taken = {}, 0, 0
    for v, l in items:
        # TODO 1: nếu taken == numWanted thì dừng (break)
        # TODO 2: nếu nhãn l còn lượt (used.get(l, 0) < useLimit) thì tăng used[l], cộng v vào total và tăng taken
        pass
    return total


# for v, l, nw, ul, e in examples:
#     assert exercise_3(v, l, nw, ul) == e

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, tự viết `exercise_4(values, labels, numWanted, useLimit)` bằng **heap** (max-heap bằng số âm).

In [31]:
def exercise_4(values, labels, numWanted, useLimit):
    # TODO
    pass


# for v, l, nw, ul, e in examples:
#     assert exercise_4(v, l, nw, ul) == e

### Bài 5 — Mini challenge: trả về các chỉ số được chọn (khó)

Viết `exercise_5(values, labels, numWanted, useLimit)` trả về **danh sách các chỉ số** (không trùng) của một tập món hợp lệ có tổng giá trị **lớn nhất**. Có thể có nhiều đáp án đúng nên kiểm bằng hàm `check_selection`. Ví dụ với ví dụ 1 của đề, một đáp án đúng là `[0, 2, 4]` (tổng `9`).

Gợi ý: sắp xếp các **chỉ số** theo `values[i]` giảm dần và duyệt như Cách 2.

In [32]:
def exercise_5(values, labels, numWanted, useLimit):
    # TODO
    pass


def check_selection(values, labels, numWanted, useLimit, chosen) -> bool:
    if chosen is None or len(set(chosen)) != len(chosen) or len(chosen) > numWanted:
        return False
    if any(not (0 <= i < len(values)) for i in chosen):
        return False
    count = {}
    for i in chosen:
        count[labels[i]] = count.get(labels[i], 0) + 1
    if any(c > useLimit for c in count.values()):
        return False
    return sum(values[i] for i in chosen) == Solution().largestValsFromLabels(values, labels, numWanted, useLimit)


# for v, l, nw, ul, e in examples:
#     assert check_selection(v, l, nw, ul, exercise_5(v, l, nw, ul))

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `sorted(zip(...), reverse=True)` sắp xếp cặp theo giá trị (phần tử đầu) giảm dần.
- Mức 2: `[(3,1), (9,2), (5,1)]` sắp xếp giảm dần theo giá trị: `9, 5, 3`.
- Mức 3: `[(9, 2), (5, 1), (3, 1)]`; `[8, 8]`; chọn `9` (nhãn `2`) và `5` (nhãn `1`): `14`.

**Bài 2**
- Mức 1: nhãn `1` có `10, 9`; nhãn `2` có `8, 7`; nhãn `3` có `6, 5`; `useLimit = 1`.
- Mức 2: duyệt `10` (nhãn `1`) chọn; `9` (nhãn `1`) bỏ; `8` (nhãn `2`) chọn; `7` bỏ; `6` (nhãn `3`) chọn; `5` bỏ.
- Mức 3: chọn `[10, 8, 6]` (chỉ `3` món dù `numWanted = 4`), tổng `24`.

**Bài 3**
- Mức 1: `if taken == numWanted: break`.
- Mức 2: `if used.get(l, 0) < useLimit:`.
- Mức 3: `used[l] = used.get(l, 0) + 1; total += v; taken += 1`.

**Bài 4**
- Mức 1: `heap = [(-v, l) for v, l in zip(values, labels)]`, `heapq.heapify(heap)`.
- Mức 2: `while heap and taken < numWanted: neg, l = heapq.heappop(heap)`.
- Mức 3: nếu `used.get(l, 0) < useLimit` thì cập nhật `used`, `total -= neg`, `taken += 1`.

**Bài 5**
- Mức 1: `order = sorted(range(n), key=lambda i: values[i], reverse=True)`.
- Mức 2: duyệt `order`, kiểm tra `taken` và `used[labels[i]]` như Cách 2.
- Mức 3: thêm `i` vào danh sách `chosen` mỗi khi chọn.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
r1 = sorted(zip([3, 9, 5], [1, 2, 1]), reverse=True)
r2 = heapq.nlargest(2, [4, 8, 1, 8])
r3 = Solution().largestValsFromLabels([3, 9, 5], [1, 2, 1], 2, 1)
print(r1, r2, r3)
assert r1 == [(9, 2), (5, 1), (3, 1)]
assert r2 == [8, 8]
assert r3 == 14

[(9, 2), (5, 1), (3, 1)] [8, 8] 14


### Đáp án Bài 2

Sắp xếp giảm dần: `10 (nhãn 1)`, `9 (1)`, `8 (2)`, `7 (2)`, `6 (3)`, `5 (3)`. Chọn `10`, bỏ `9`, chọn `8`, bỏ `7`, chọn `6`, bỏ `5`: ba món vì `useLimit = 1` và có ba nhãn. Tổng `24`.

In [34]:
hand_chosen_solution, hand_answer_solution = [10, 8, 6], 24
v, l = [10, 9, 8, 7, 6, 5], [1, 1, 2, 2, 3, 3]
assert hand_answer_solution == sum(hand_chosen_solution) == sol(v, l, 4, 1) == largest_brute(v, l, 4, 1)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(values, labels, numWanted, useLimit):
    items = sorted(zip(values, labels), reverse=True)
    used, total, taken = {}, 0, 0
    for v, l in items:
        if taken == numWanted:
            break
        if used.get(l, 0) < useLimit:
            used[l] = used.get(l, 0) + 1
            total += v
            taken += 1
    return total

for v, l, nw, ul, e in examples:
    assert exercise_3_solution(v, l, nw, ul) == e
rng = random.Random(3)
for _ in range(500):
    n = rng.randint(1, 10)
    v = [rng.randint(0, 9) for _ in range(n)]
    l = [rng.randint(0, 3) for _ in range(n)]
    nw, ul = rng.randint(1, n), rng.randint(1, n)
    assert exercise_3_solution(v, l, nw, ul) == largest_brute(v, l, nw, ul)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(values, labels, numWanted, useLimit):
    heap = [(-v, l) for v, l in zip(values, labels)]
    heapq.heapify(heap)
    used, total, taken = {}, 0, 0
    while heap and taken < numWanted:
        neg, l = heapq.heappop(heap)
        if used.get(l, 0) < useLimit:
            used[l] = used.get(l, 0) + 1
            total -= neg
            taken += 1
    return total

for v, l, nw, ul, e in examples:
    assert exercise_4_solution(v, l, nw, ul) == e
rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 10)
    v = [rng.randint(0, 9) for _ in range(n)]
    l = [rng.randint(0, 3) for _ in range(n)]
    nw, ul = rng.randint(1, n), rng.randint(1, n)
    assert exercise_4_solution(v, l, nw, ul) == largest_brute(v, l, nw, ul)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(values, labels, numWanted, useLimit):
    order = sorted(range(len(values)), key=lambda i: values[i], reverse=True)
    used, chosen = {}, []
    for i in order:
        if len(chosen) == numWanted:
            break
        if used.get(labels[i], 0) < useLimit:
            used[labels[i]] = used.get(labels[i], 0) + 1
            chosen.append(i)
    return chosen

def check_selection_solution(values, labels, numWanted, useLimit, chosen):
    if chosen is None or len(set(chosen)) != len(chosen) or len(chosen) > numWanted:
        return False
    count = {}
    for i in chosen:
        count[labels[i]] = count.get(labels[i], 0) + 1
    if any(c > useLimit for c in count.values()):
        return False
    return sum(values[i] for i in chosen) == largest_brute(values, labels, numWanted, useLimit)

for v, l, nw, ul, e in examples:
    assert check_selection_solution(v, l, nw, ul, exercise_5_solution(v, l, nw, ul))
rng = random.Random(5)
for _ in range(500):
    n = rng.randint(1, 10)
    v = [rng.randint(0, 9) for _ in range(n)]
    l = [rng.randint(0, 3) for _ in range(n)]
    nw, ul = rng.randint(1, n), rng.randint(1, n)
    assert check_selection_solution(v, l, nw, ul, exercise_5_solution(v, l, nw, ul)), (v, l, nw, ul)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (tham lam: giá trị giảm dần + đếm nhãn)

```python
class Solution:
    def largestValsFromLabels(self, values: List[int], labels: List[int], numWanted: int, useLimit: int) -> int:
        items = sorted(zip(values, labels), reverse=True)    # giữ cặp (giá trị, nhãn)
        used = {}
        total = taken = 0
        for v, l in items:
            if taken == numWanted:
                break
            if used.get(l, 0) < useLimit:
                used[l] = used.get(l, 0) + 1
                total += v
                taken += 1
        return total
```

Công thức nhớ nhanh: **sắp xếp `(giá trị, nhãn)` giảm dần; chọn món nếu nhãn còn lượt (`used < useLimit`); dừng khi đủ `numWanted`**.

### Checklist trước khi submit

- [ ] Sắp xếp theo **cặp** `(giá trị, nhãn)` (không sắp xếp riêng `values` và `labels`)?
- [ ] Điều kiện còn lượt là `used.get(l, 0) < useLimit` (nghiêm ngặt)?
- [ ] Dừng khi đã chọn đủ `numWanted` món (`taken == numWanted`)?
- [ ] Chỉ đúng khi giá trị **không âm** (nếu có giá trị âm thì dừng ở giá trị `<= 0`)?
- [ ] Đã thử: một nhãn, mọi nhãn khác nhau, `numWanted = 1`, `useLimit = 1`, mọi giá trị `0`, `n = 2 * 10^4`?